In [1]:
import os
import gdstk
import svgutils.transform as sg
import ipywidgets as widgets
from IPython.display import SVG, display, clear_output
from glayout import gf180
from glayout.cells.composite.opamp.opamp import opamp

def display_gds(gds_file, path, scale=3):
    top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
    top_level_cell.write_svg(os.path.join(path, 'out.svg'))
    fig = sg.fromfile(os.path.join(path, 'out.svg'))
    fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
    fig.save(os.path.join(path, 'out.svg'))
    display(SVG(os.path.join(path, 'out.svg')))

def display_component(component, path, scale=3):
    with hide:
        component.write_gds(os.path.join(path, 'out.gds'))
    display_gds(os.path.join(path, 'out.gds'), path, scale)

In [2]:
pdk = gf180

half_diffpair_params = (6, 1, 4)
diffpair_bias = (6, 2, 4)
half_common_source_params = (7, 1, 10, 3)
half_common_source_bias = (6, 2, 8, 2)
output_stage_params = (5, 1, 16)
output_stage_bias = (6, 2, 4)
half_pload = (6, 1, 6)
mim_cap_size = (12, 12)
mim_cap_rows = 3
rmult = 2

hide = widgets.Output()

print('Generating Op-Amp Variant 1...')
component = opamp(
    pdk, 
    half_diffpair_params, 
    diffpair_bias, 
    half_common_source_params,
    half_common_source_bias, 
    output_stage_params,
    output_stage_bias,
    half_pload,
    mim_cap_size,
    mim_cap_rows,
    rmult
)

clear_output()
display_component(component, '/foss/designs/sscs-2026-zotnetic/designs/notebooks/', 0.5)


In [3]:
print(component.info['netlist'].generate_netlist())

.subckt NMOS D G S B DUM l=1.0 w=6.0
XMAIN1 D G S B nfet_03v3 l=1.0 w=6.0
XMAIN2 D G S B nfet_03v3 l=1.0 w=6.0
XMAIN3 D G S B nfet_03v3 l=1.0 w=6.0
XMAIN4 D G S B nfet_03v3 l=1.0 w=6.0
XDUMMY1 DUM DUM DUM B nfet_03v3 l=1.0 w=6.0
.ends NMOS

.subckt DIFF_PAIR VP VN VDD1 VDD2 VTAIL B
X0 VDD1 VP VTAIL B B NMOS l=1.0 w=6.0
X1 VDD1 VP VTAIL B B NMOS l=1.0 w=6.0
X2 VDD2 VN VTAIL B B NMOS l=1.0 w=6.0
X3 VDD2 VN VTAIL B B NMOS l=1.0 w=6.0
.ends DIFF_PAIR

.subckt two_trans_interdigitized VDD1 VDD2 VSS1 VSS2 VG1 VG2 VB l=2.0 w=6.0 m=1 
XA VDD1 VG1 VSS1 VB nfet_03v3 l=2.0 w=6.0 m=4
XB VDD2 VG2 VSS2 VB nfet_03v3 l=2.0 w=6.0 m=4
XDUMMY cmdum cmdum cmdum VB nfet_03v3 l=2.0 w=6.0 m=2
.ends two_trans_interdigitized

.subckt CMIRROR VREF VOUT VSS B
X0 VREF VOUT VSS VSS VREF VREF B two_trans_interdigitized l=2.0 w=6.0 m={1}
.ends CMIRROR

.subckt NMOS_1 D G S B DUM l=0.5 w=1
XMAIN1 D G S B nfet_03v3 l=0.5 w=1
XMAIN2 D G S B nfet_03v3 l=0.5 w=1
XMAIN3 D G S B nfet_03v3 l=0.5 w=1
XMAIN4 D G S B nfet_03v3

In [5]:
netlist = component.info['netlist'].generate_netlist()
with open('/foss/designs/sscs-2026-zotnetic/designs/notebooks/opamp.spice', 'w') as f: 
    f.write(netlist)
print('Netlist saved!')

Netlist saved!
